In [ ]:
from huggingface_hub import HfApi

api = HfApi()
api.upload_folder(
    folder_path="../sroie-receipt-dataset",  # change this to your actual folder path
    repo_id="SahilSheth1/sroie-receipt-dataset",  # change to your HF username
    repo_type="dataset"
)

In [ ]:
import os
import json
import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from collections import defaultdict

In [ ]:
import sys, platform
print("Python:", sys.executable)
print("Architecture:", platform.machine())
print("Version:", sys.version)

import pandas as pd
import numpy as np
print("pandas:", pd.__version__)
print("numpy:", np.__version__)

In [ ]:
BASE     = "../sroie-receipt-dataset/SROIE2019"
SPLITS   = ["train", "test"]

In [ ]:
records = []
for split in SPLITS:
    img_dir = os.path.join(BASE, split, "img")
    ent_dir = os.path.join(BASE, split, "entities")

    images      = sorted([f for f in os.listdir(img_dir) if f.endswith(".jpg")])
    entity_files = sorted([f for f in os.listdir(ent_dir)])

    print(f"\n{split.upper()}: {len(images)} images | {len(entity_files)} entity files")

    for img_file in images:
        stem = os.path.splitext(img_file)[0]
        ent_path = os.path.join(ent_dir, stem + ".txt")

        # image dimensions
        img = Image.open(os.path.join(img_dir, img_file))
        w, h = img.size

        # entity fields
        fields = {"company": None, "date": None, "address": None, "total": None}

        # Replace the entity parsing section with this:
        if os.path.exists(ent_path):
            with open(ent_path, "r", encoding="utf-8", errors="ignore") as f:
                try:
                    data = json.load(f)
                    for key in fields:
                        fields[key] = data.get(key, None) or None
                except json.JSONDecodeError:
                    pass  # leave fields as None if file is malformed

        records.append({"split": split, "file": img_file,
                         "width": w, "height": h, **fields})

df = pd.DataFrame(records)
print("\n── Dataset shape:", df.shape)
print(df.head(3))

In [ ]:
print("\n── Image Dimensions ──")
print(df[["width","height"]].describe().round(1))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(df["width"],  bins=30, color="steelblue", edgecolor="white")
axes[0].set_title("Width Distribution");  axes[0].set_xlabel("px")
axes[1].hist(df["height"], bins=30, color="coral",     edgecolor="white")
axes[1].set_title("Height Distribution"); axes[1].set_xlabel("px")
plt.tight_layout()
plt.savefig("EDA_Results/eda_dimensions.png", dpi=120)
plt.show()

In [ ]:
fields = ["company","date","address","total"]
completeness = df[fields].notnull().mean() * 100

print("\n── Field Completeness (%) ──")
print(completeness.round(1))

completeness.plot(kind="bar", color=["#4C72B0","#DD8452","#55A868","#C44E52"],
                  edgecolor="white", figsize=(8,4))
plt.title("Field Completeness Across Full Dataset")
plt.ylabel("% receipts with field present")
plt.xticks(rotation=0); plt.ylim(0,110)
plt.tight_layout()
plt.savefig("EDA_Results/eda_completeness.png", dpi=120)
plt.show()

In [ ]:
all_four = df[fields].notnull().all(axis=1).sum()
print(f"\n── Receipts with ALL 4 fields: {all_four} / {len(df)} "
      f"({100*all_four/len(df):.1f}%)")

In [ ]:
df["missing"] = df[fields].isnull().apply(
    lambda r: ", ".join([f for f in fields if pd.isnull(r[f])]) or "none", axis=1)
print("\n── Missing field patterns ──")
print(df["missing"].value_counts().head(10))

In [ ]:
samples = df[df["split"]=="train"].sample(4, random_state=42)
fig, axes = plt.subplots(1, 4, figsize=(16, 6))
for ax, (_, row) in zip(axes, samples.iterrows()):
    img = Image.open(os.path.join(BASE, row["split"], "img", row["file"]))
    ax.imshow(img, cmap="gray")
    ax.set_title(f"W:{row['width']} H:{row['height']}", fontsize=8)
    ax.axis("off")
plt.suptitle("Sample Training Receipts")
plt.tight_layout()
plt.savefig("EDA_Results/eda_samples.png", dpi=120)
plt.show()

print("\nEDA complete — 3 figures saved.")

In [ ]:
from sklearn.model_selection import train_test_split

SEED = 42

df_train_full = df[df["split"] == "train"].reset_index(drop=True)
df_test       = df[df["split"] == "test"].reset_index(drop=True)

df_train, df_val = train_test_split(df_train_full, test_size=0.1, random_state=SEED)

print(f"Train : {len(df_train)}")
print(f"Val   : {len(df_val)}")
print(f"Test  : {len(df_test)}")

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

class SROIEDataset(Dataset):
    TRANSFORM = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.Grayscale(num_output_channels=3),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                             std=[0.229, 0.224, 0.225])
    ])

    def __init__(self, dataframe, base_path, transform=None):
        self.df        = dataframe.reset_index(drop=True)
        self.base_path = base_path
        self.transform = transform or self.TRANSFORM

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row      = self.df.iloc[idx]
        img_path = os.path.join(self.base_path, row["split"], "img", row["file"])
        image    = Image.open(img_path).convert("RGB")
        image    = self.transform(image)

        annotation = {
            "company" : row["company"] or "",
            "date"    : row["date"]    or "",
            "address" : row["address"] or "",
            "total"   : row["total"]   or "",
        }
        return image, annotation

train_dataset = SROIEDataset(df_train, BASE)
val_dataset   = SROIEDataset(df_val,   BASE)
test_dataset  = SROIEDataset(df_test,  BASE)

print(f"Train: {len(train_dataset)} | Val: {len(val_dataset)} | Test: {len(test_dataset)}")

In [ ]:
def collate_fn(batch):
    images      = torch.stack([item[0] for item in batch])
    annotations = [item[1] for item in batch]
    return images, annotations

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True,
                          num_workers=0, collate_fn=collate_fn)
val_loader   = DataLoader(val_dataset,   batch_size=16, shuffle=False,
                          num_workers=0, collate_fn=collate_fn)
test_loader  = DataLoader(test_dataset,  batch_size=16, shuffle=False,
                          num_workers=0, collate_fn=collate_fn)

images, annotations = next(iter(train_loader))
print(f"Batch shape : {images.shape}")        # expect [16, 3, 224, 224]
print(f"Sample annotation: {annotations[0]}")

In [ ]:
# Cell 12 — Tesseract + Regex Baseline (Experiment 1)
import sys
sys.path.append("../src")
from baseline import run_baseline

# Build test samples in the format baseline.py expects
test_samples = []
for _, row in df_test.iterrows():
    test_samples.append({
        "image_path": os.path.join(BASE, row["split"], "img", row["file"]),
        "annotation": {
            "company" : row["company"] or "",
            "date"    : row["date"]    or "",
            "address" : row["address"] or "",
            "total"   : row["total"]   or "",
        }
    })

# Run baseline — takes a few minutes (Tesseract processes every image)
results = run_baseline(test_samples)

# Print results
print("\n===== EXPERIMENT 1: TESSERACT + REGEX BASELINE =====")
for field, metrics in results.items():
    print(f"{field.upper():10s}  Exact Match: {metrics['exact_match']:.3f}  F1: {metrics['f1']:.3f}")

# Save for experiment log
with open("../Experiments/experiment1_results.json", "w") as f:
    json.dump(results, f, indent=2)
print("\nSaved to experiments/experiment1_results.json")